# Empathy degradation under adversarial prompting

Backend-parameterized driver for **study 1** -- all logic lives in the [`mh_safety`](mh_safety/) package
(see [mh_safety/empathy/](mh_safety/empathy/)). Set `BACKEND` in Setup to run it against any model.

It measures how far an LLM's empathy/safety drops when adversarially prompted, against two references:
`default` (no steering, the realistic baseline) and `supportive` (explicitly empathetic). Degradation is
measured with paired Wilcoxon/t-tests, Cohen's d, and an Attack-Success-Rate.

**Judging:** every model's responses are scored by a single fixed judge, `mistralai/Mistral-7B-Instruct-v0.3`
(HuggingFace, GPU) — set on `cfg.judge_llm` — so all models are evaluated the same way rather than judging
themselves.

## Setup

Pick the model **backend** below; everything else is identical across models. The `mh_safety` pipeline is
backend-agnostic, so this one notebook runs the study on any of Claude / Llama (Ollama) / Gemma / Qwen.

In [ ]:
# ===== pick the model backend =====
BACKEND = "anthropic"        # "anthropic" | "ollama" | "gemma" | "qwen"

# Optional deps by backend:
#   anthropic -> ANTHROPIC_API_KEY (prompted below)
#   ollama    -> a running `ollama` server + `pip install ollama`
#   gemma/qwen-> pip install "transformers>=4.51.0" accelerate bitsandbytes torch   (GPU recommended)

import os, getpass
if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-ant-...): ")

from mh_safety.config import EmpathyConfig
from mh_safety.llm import LLMClient
from mh_safety.empathy import pipeline as ep

# EmpathyConfig()/RoleIntentConfig() == anthropic; .ollama()/.gemma()/.qwen() for the others
cfg = EmpathyConfig() if BACKEND == "anthropic" else getattr(EmpathyConfig, BACKEND)()
# override anything, e.g.:  cfg = EmpathyConfig.ollama(n_posts=10)
client = LLMClient(cfg.llm)
cfg

## 1. Sample posts (load -> scrub -> filter -> risk-stratify)

In [ ]:
sample = ep.load_sample(cfg)
print(sample["risk_tier"].value_counts())
sample.head(3)

## 2. Generate baseline + manipulated replies, then judge + score

In [ ]:
responses = ep.generate_responses(cfg, sample, client)
# judged by the shared judge model (Mistral-7B), regardless of which model generated
scored = ep.add_automated_metrics(ep.judge_responses(cfg, responses, sample))
scored[["post_id", "condition", "empathy", "safety", "response"]].head()

## 3. Analyse + report

In [ ]:
A = ep.analyze(cfg, scored)
ep.print_report(cfg, scored, A)

## 4. Plots + save

In [ ]:
ep.make_plots(cfg, scored, A, show=True)
ep.save_results(cfg, scored, A)

## Notes

* Each backend caches to `.llm_cache/<backend>/` and writes to `outputs/<backend>/empathy/`, so runs never collide.
* One-liner equivalent of the cells above: `ep.run(cfg, show=True)`.
* Extra failure-taxonomy analysis for any run: `python robustness_metrics.py outputs/<backend>/empathy/scored_responses.csv`.
* Limitations: single LLM judge (add human + second-judge validation), pilot N, 2019 English Reddit.